In [9]:
#New RAG Model
# radar_rag_engine.py
# ---------------------------------------------------------------
# Offline Radar Telemetry Interpreter
# Requires: Ollama running locally with Qwen model downloaded
# ---------------------------------------------------------------

import json
import time
import sys
from typing import List, Dict, Any, Optional
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
import requests
import pandas as pd

# ---------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5-coder:7b-instruct-q5_K_M"

registry = get_registry().get("sentence-transformers")
embedding_model = registry.create(name="BAAI/bge-small-en-v1.5")


def call_local_qwen(prompt: str, json_mode: bool = False) -> str:
    """
    Call the local Qwen LLM via Ollama API
    
    Args:
        prompt: The input prompt for the LLM
        json_mode: If True, request JSON-formatted response
        
    Returns:
        LLM response text or error message
    """
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0}
    }
    if json_mode:
        payload["format"] = "json"

    try:
        print("[*] Waiting for LLM response (this may take 10-30 seconds)...")
        response = requests.post(OLLAMA_URL, json=payload, timeout=300)
        if response.status_code == 200:
            return response.json().get("response", "").strip()
        else:
            return f"Error: Ollama returned status {response.status_code}"
    except requests.exceptions.Timeout:
        return "Error: Ollama request timed out (300s). Try again or check if Ollama is running."
    except requests.exceptions.ConnectionError:
        return "Error: Cannot connect to Ollama at http://localhost:11434. Make sure Ollama is running."
    except Exception as e:
        return f"Error connecting to local LLM: {str(e)}"


class RadarLogSchema(LanceModel):
    """
    Schema for radar telemetry logs stored in LanceDB
    
    Attributes:
        id: Unique identifier for the log entry
        timestamp: ISO format timestamp (YYYY-MM-DD HH:MM:SS)
        radar_id: Identifier for the radar unit (e.g., "AN-FPS-117_A")
        azimuth: Horizontal angle in degrees (0-360)
        elevation: Vertical angle in degrees (-90 to +90)
        voltage_mv: Power supply voltage in millivolts
        error_code: Status code ("OK" or "ERR_XXX_DESCRIPTION")
        log_message: Full text description of the event
        vector: 384-dimensional embedding of log_message
    """
    id: str
    timestamp: str
    radar_id: str
    azimuth: float
    elevation: float
    voltage_mv: int
    error_code: str
    log_message: str
    vector: Vector(384) = embedding_model.VectorField()


try:
    db = lancedb.connect("./local_radar_db")
    table_name = "radar_telemetry"

    if table_name in db.table_names():
        table = db.open_table(table_name)
    else:
        table = db.create_table(table_name, schema=RadarLogSchema)
except Exception as e:
    print(f"[ERROR] Failed to connect to LanceDB: {str(e)}")
    raise


class ContinuumMemory:
    """
    Persistent memory system for tracking radar faults and anomalies across sessions
    
    Stores:
        - Last processed timestamp
        - Known critical faults
        - Active anomalies per radar unit
    """
    def __init__(self):
        self.state_file = "radar_continuum_memory.json"
        self.initialize_memory()

    def initialize_memory(self):
        """Load memory from disk or create new state"""
        try:
            with open(self.state_file, 'r') as f:
                self.memory = json.load(f)
        except FileNotFoundError:
            self.memory = {
                "last_processed_timestamp": "1970-01-01 00:00:00",
                "known_critical_faults": [],
                "active_anomalies": {}
            }
            self.save()
        except json.JSONDecodeError:
            print(f"[WARNING] Memory file corrupted, starting fresh")
            self.memory = {
                "last_processed_timestamp": "1970-01-01 00:00:00",
                "known_critical_faults": [],
                "active_anomalies": {}
            }
            self.save()

    def save(self):
        """Persist memory state to JSON file"""
        try:
            with open(self.state_file, 'w') as f:
                json.dump(self.memory, f, indent=2)
        except Exception as e:
            print(f"[WARNING] Failed to save memory: {str(e)}")

    def update_state(self, new_fault: str, radar_id: str):
        """
        Add new fault to history and update anomaly tracking
        
        Args:
            new_fault: Error code to track
            radar_id: Radar unit identifier
        """
        if new_fault not in self.memory["known_critical_faults"]:
            self.memory["known_critical_faults"].append(new_fault)
        self.memory["active_anomalies"][radar_id] = f"Last fault tracked: {new_fault}"
        self.save()

    def get_context_string(self) -> str:
        """Export memory as JSON string for LLM context"""
        return json.dumps(self.memory)


# ---------------------------------------------------------------
# ERROR CODE REFERENCE
# ---------------------------------------------------------------

ERROR_CODE_REFERENCE = {
    "OK": {
        "description": "Normal operation",
        "severity": "🟢 NORMAL",
        "action": "No action required"
    },
    "ERR_403_UNDERVOLT": {
        "description": "Power supply voltage dropped below safe operating margins",
        "severity": "🔴 CRITICAL",
        "action": "Check power supply connections and voltage regulator. May require immediate system shutdown."
    },
    "ERR_501_ALIGN_FAIL": {
        "description": "Mechanical misalignment detected in antenna or gimbal system",
        "severity": "🟠 HIGH",
        "action": "Perform mechanical alignment check. Contact maintenance for calibration."
    },
    "ERR_601_THERMAL_HIGH": {
        "description": "Internal temperature exceeds safe operating limits",
        "severity": "🟠 HIGH",
        "action": "Check cooling system and ventilation. Reduce operational load if necessary."
    },
    "ERR_701_MOTOR_FAULT": {
        "description": "Motor malfunction or bearing failure detected",
        "severity": "🟠 HIGH",
        "action": "Inspect motor assembly. Schedule motor replacement."
    },
    "ERR_801_COMMS_LOSS": {
        "description": "Communication link to control station lost",
        "severity": "🟠 HIGH",
        "action": "Verify network connectivity and signal strength. Restart communication module."
    },
    "ERR_900_UNKNOWN": {
        "description": "Unknown or undefined error",
        "severity": "🟡 MEDIUM",
        "action": "Enable debug logging and contact support."
    }
}


def print_input_format():
    """
    Display expected input data format for telemetry records
    """
    print("\n" + "="*70)
    print("📋 EXPECTED INPUT DATA FORMAT")
    print("="*70)
    
    print("\n--- SINGLE RECORD FORMAT (Interactive Input) ---")
    print("""
Radar ID:           AN-FPS-117_A
Timestamp:          2026-05-22 14:00:00
Azimuth:            45.2
Elevation:          12.1
Voltage (mV):       5000
Error Code:         OK or ERR_403_UNDERVOLT
Log Message:        Normal sweep operations. Thermal gradients stable.
    """)
    
    print("\n--- JSON ARRAY FORMAT (File Input) ---")
    print("""
[
    {
        "id": "AN-FPS-117_A_2026-05-22_14-00-00",
        "timestamp": "2026-05-22 14:00:00",
        "radar_id": "AN-FPS-117_A",
        "azimuth": 45.2,
        "elevation": 12.1,
        "voltage_mv": 5000,
        "error_code": "OK",
        "log_message": "Normal sweep operations. Thermal gradients stable within threshold limits."
    },
    {
        "id": "AN-FPS-117_A_2026-05-22_14-02-15",
        "timestamp": "2026-05-22 14:02:15",
        "radar_id": "AN-FPS-117_A",
        "azimuth": 89.7,
        "elevation": 11.5,
        "voltage_mv": 3100,
        "error_code": "ERR_403_UNDERVOLT",
        "log_message": "Critical Alert: Shifter power rail bus voltage dropped below safe margins."
    }
]
    """)
    
    print("\n--- FIELD SPECIFICATIONS ---")
    print("""
┌─────────────────┬──────────────────────┬──────────────────────────────┐
│ Field           │ Type                 │ Valid Values / Format        │
├─────────────────┼──────────────────────┼──────────────────────────────┤
│ id              │ String               │ Unique identifier (auto)     │
│ timestamp       │ String (YYYY-MM-DD   │ ISO 8601 format              │
│                 │ HH:MM:SS)            │ e.g., "2026-05-22 14:00:00"  │
│ radar_id        │ String               │ e.g., "AN-FPS-117_A"         │
│ azimuth         │ Float                │ 0.0 to 360.0 degrees        │
│ elevation       │ Float                │ -90.0 to +90.0 degrees      │
│ voltage_mv      │ Integer              │ 0 to 9999 millivolts         │
│ error_code      │ String               │ "OK" or "ERR_NNN_DESCRIPTION"│
│ log_message     │ String               │ Text description (any length)│
└─────────────────┴──────────────────────┴──────────────────────────────┘
    """)
    
    print("\n--- EXAMPLE ERROR CODES ---")
    print("""
OK                      → Normal operation
ERR_403_UNDERVOLT       → Power supply voltage low
ERR_501_ALIGN_FAIL      → Mechanical misalignment detected
ERR_601_THERMAL_HIGH    → Temperature exceeds safe limits
ERR_701_MOTOR_FAULT     → Motor malfunction
ERR_801_COMMS_LOSS      → Communication loss
    """)
    
    print("="*70 + "\n")


def add_telemetry_record(radar_id: str, timestamp: str, azimuth: float, elevation: float,
                         voltage_mv: int, error_code: str, log_message: str) -> bool:
    """
    Add a single telemetry record to the database
    
    Args:
        radar_id: Unit identifier
        timestamp: ISO format (YYYY-MM-DD HH:MM:SS)
        azimuth: Horizontal angle (0-360)
        elevation: Vertical angle (-90 to +90)
        voltage_mv: Power voltage
        error_code: Status code
        log_message: Event description
        
    Returns:
        True if successful, False otherwise
    """
    try:
        record = {
            "id": f"{radar_id}_{timestamp.replace(' ', '_').replace(':', '-')}",
            "timestamp": timestamp,
            "radar_id": radar_id,
            "azimuth": azimuth,
            "elevation": elevation,
            "voltage_mv": voltage_mv,
            "error_code": error_code,
            "log_message": log_message
        }
        table.add([record])
        print(f"[✔] Record added: {radar_id} @ {timestamp}")
        return True
    except Exception as e:
        print(f"[ERROR] Failed to add record: {str(e)}")
        return False


def bulk_add_telemetry(telemetry_list: List[Dict]) -> bool:
    """
    Add multiple telemetry records to the database
    
    Args:
        telemetry_list: List of telemetry dictionaries
        
    Returns:
        True if successful, False otherwise
    """
    try:
        table.add(telemetry_list)
        print(f"[✔] {len(telemetry_list)} records added to database")
        return True
    except Exception as e:
        print(f"[ERROR] Bulk add failed: {str(e)}")
        return False


def hybrid_retrieve(query_text: str, time_filter: Optional[str] = None, limit: int = 3, retry: int = 0) -> List[Dict]:
    """
    Retrieve relevant telemetry records from database
    
    Uses pandas-based filtering to search across multiple fields
    Supports optional time-range filtering
    
    Args:
        query_text: Search query text
        time_filter: Optional timestamp prefix filter (YYYY-MM-DD)
        limit: Maximum records to return
        retry: Current retry attempt (not used, kept for compatibility)
        
    Returns:
        List of matching telemetry records
    """
    try:
        # Get all data from LanceDB
        all_data = table.to_pandas()
        
        if all_data.empty:
            print("[*] Database is empty")
            return []
        
        # Print debug info
        print(f"[DEBUG] Database has {len(all_data)} records")
        print(f"[DEBUG] Searching for: '{query_text}'")
        
        # Search in ALL text fields, not just log_message
        query_lower = query_text.lower()
        
        # Create comprehensive search mask
        mask = (
            all_data['log_message'].str.lower().str.contains(query_lower, na=False) |
            all_data['error_code'].str.lower().str.contains(query_lower, na=False) |
            all_data['radar_id'].str.lower().str.contains(query_lower, na=False)
        )
        
        if time_filter:
            mask = mask & all_data['timestamp'].str.startswith(time_filter)
        
        filtered_results = all_data[mask].head(limit).to_dict('records')
        
        print(f"[DEBUG] Found {len(filtered_results)} matches")
        
        # If no results, return all records for analysis (fallback)
        if not filtered_results:
            print(f"[*] No exact matches found, returning all records for analysis")
            return all_data.head(limit).to_dict('records')
        
        return [
            {
                "timestamp": r.get("timestamp", ""),
                "radar_id": r.get("radar_id", ""),
                "azimuth": r.get("azimuth", 0),
                "elevation": r.get("elevation", 0),
                "voltage": r.get("voltage_mv", 0),
                "error_code": r.get("error_code", ""),
                "message": r.get("log_message", "")
            } for r in filtered_results
        ]
    except Exception as e:
        print(f"[ERROR] hybrid_retrieve failed: {str(e)}")
        # Fallback: return first 3 records
        try:
            all_data = table.to_pandas()
            results = all_data.head(3).to_dict('records')
            return [
                {
                    "timestamp": r.get("timestamp", ""),
                    "radar_id": r.get("radar_id", ""),
                    "azimuth": r.get("azimuth", 0),
                    "elevation": r.get("elevation", 0),
                    "voltage": r.get("voltage_mv", 0),
                    "error_code": r.get("error_code", ""),
                    "message": r.get("log_message", "")
                } for r in results
            ]
        except:
            return []


def generate_human_readable_summary(memory_engine: ContinuumMemory) -> str:
    """
    Generate a human-readable summary of all telemetry data
    
    Args:
        memory_engine: ContinuumMemory instance for context
        
    Returns:
        Human-friendly summary text
    """
    try:
        all_data = table.to_pandas()
        
        if all_data.empty:
            return "[*] No telemetry data available in database."
        
        # Prepare data summary
        summary_data = {
            "total_records": len(all_data),
            "radar_units": all_data['radar_id'].unique().tolist(),
            "time_range": f"{all_data['timestamp'].min()} to {all_data['timestamp'].max()}",
            "error_codes": all_data['error_code'].unique().tolist(),
            "avg_voltage": all_data['voltage_mv'].mean(),
            "min_voltage": all_data['voltage_mv'].min(),
            "max_voltage": all_data['voltage_mv'].max(),
            "records_sample": all_data.head(5).to_dict('records')
        }
        
        # LLM prompt for human-readable summary
        summary_prompt = f"""You are a radar telemetry analyst. Generate a clear, human-readable summary of the radar system operations.

[System Memory]: {memory_engine.get_context_string()}

[Telemetry Summary]:
- Total Records: {summary_data['total_records']}
- Radar Units: {summary_data['radar_units']}
- Time Period: {summary_data['time_range']}
- Error Codes Present: {summary_data['error_codes']}
- Voltage Stats (mV): Min={summary_data['min_voltage']}, Max={summary_data['max_voltage']}, Avg={summary_data['avg_voltage']:.1f}

[Sample Records]:
{json.dumps(summary_data['records_sample'], indent=2)}

Generate a clear, concise English summary highlighting:
1. Overall system status
2. Key events and anomalies
3. Performance metrics
4. Any critical issues

Keep it professional but accessible to non-technical readers."""

        response = call_local_qwen(summary_prompt)
        
        if response.startswith("Error"):
            return f"[ERROR] LLM summary generation failed: {response}\n\nBasic Stats:\n{json.dumps(summary_data, indent=2, default=str)}"
        
        return response
        
    except Exception as e:
        return f"[ERROR] Summary generation failed: {str(e)}"


def display_error_code_details():
    """
    Display detailed information about all error codes found in database
    """
    try:
        all_data = table.to_pandas()
        
        if all_data.empty:
            print("[*] No data in database")
            return
        
        error_codes = all_data['error_code'].unique()
        error_counts = all_data['error_code'].value_counts().to_dict()
        
        print("\n" + "="*80)
        print("🔍 ERROR CODE TECHNICAL REFERENCE")
        print("="*80)
        
        for error_code in sorted(error_codes):
            count = error_counts.get(error_code, 0)
            ref = ERROR_CODE_REFERENCE.get(error_code, ERROR_CODE_REFERENCE["ERR_900_UNKNOWN"])
            
            print(f"\n[{error_code}] (Occurrences: {count})")
            print(f"├─ Severity: {ref['severity']}")
            print(f"├─ Description: {ref['description']}")
            print(f"└─ Recommended Action: {ref['action']}")
        
        print("\n" + "="*80 + "\n")
        
    except Exception as e:
        print(f"[ERROR] Failed to display error codes: {str(e)}")


def display_database_stats():
    """Display current database statistics"""
    try:
        all_data = table.to_pandas()
        print("\n" + "="*60)
        print("📊 DATABASE STATISTICS")
        print("="*60)
        print(f"Total Records: {len(all_data)}")
        if not all_data.empty:
            print(f"Radar Units: {all_data['radar_id'].unique().tolist()}")
            print(f"Error Codes Found: {all_data['error_code'].unique().tolist()}")
            print(f"Time Range: {all_data['timestamp'].min()} to {all_data['timestamp'].max()}")
        print("="*60 + "\n")
    except Exception as e:
        print(f"[ERROR] Failed to display stats: {str(e)}")


def interactive_input_mode():
    """
    Interactive mode for users to input telemetry records one by one
    """
    print("\n" + "="*60)
    print("🔧 INTERACTIVE TELEMETRY INPUT MODE")
    print("="*60)
    print("Enter telemetry data record by record.")
    print("Type 'done' when finished, 'skip' to cancel current record.")
    print("Type 'format' to display expected input format.\n")
    
    records = []
    record_count = 1
    
    while True:
        print(f"\n--- Record #{record_count} ---")
        
        try:
            radar_id = input("Radar ID (e.g., AN-FPS-117_A) [or 'done']: ").strip()
            if radar_id.lower() == "done":
                break
            if radar_id.lower() == "skip":
                print("[*] Record skipped")
                continue
            if radar_id.lower() == "format":
                print_input_format()
                continue
            
            timestamp = input("Timestamp (YYYY-MM-DD HH:MM:SS): ").strip()
            azimuth = float(input("Azimuth (0-360): ").strip())
            elevation = float(input("Elevation (-90 to +90): ").strip())
            voltage_mv = int(input("Voltage (mV): ").strip())
            error_code = input("Error Code (OK or ERR_XXX_DESCRIPTION): ").strip()
            log_message = input("Log Message: ").strip()
            
            record = {
                "id": f"{radar_id}_{timestamp.replace(' ', '_').replace(':', '-')}",
                "timestamp": timestamp,
                "radar_id": radar_id,
                "azimuth": azimuth,
                "elevation": elevation,
                "voltage_mv": voltage_mv,
                "error_code": error_code,
                "log_message": log_message
            }
            records.append(record)
            print(f"[✔] Record added")
            record_count += 1
            
        except ValueError as e:
            print(f"[ERROR] Invalid input: {str(e)}. Please try again.")
            continue
        except KeyboardInterrupt:
            print("\n[*] Input cancelled")
            break
    
    if records:
        if bulk_add_telemetry(records):
            print(f"\n[✔] Successfully added {len(records)} records")
            time.sleep(1)
    
    return len(records) > 0


def json_file_input_mode():
    """
    Load telemetry data from a JSON file
    """
    print("\n" + "="*60)
    print("📄 JSON FILE INPUT MODE")
    print("="*60)
    print("Expected format: Array of telemetry records")
    print("Type 'format' to display expected format.\n")
    
    file_input = input("Enter JSON file path [or 'format']: ").strip()
    
    if file_input.lower() == "format":
        print_input_format()
        return False
    
    file_path = file_input
    
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        # Handle both single dict and list of dicts
        if isinstance(data, dict):
            records = [data]
        elif isinstance(data, list):
            records = data
        else:
            print("[ERROR] JSON must be dict or list of dicts")
            return False
        
        if bulk_add_telemetry(records):
            print(f"\n[✔] Successfully loaded {len(records)} records from {file_path}")
            return True
        else:
            return False
            
    except FileNotFoundError:
        print(f"[ERROR] File not found: {file_path}")
        return False
    except json.JSONDecodeError as e:
        print(f"[ERROR] Invalid JSON: {str(e)}")
        return False
    except Exception as e:
        print(f"[ERROR] Failed to load file: {str(e)}")
        return False


def process_radar_inquiry(user_question: str, memory_engine: ContinuumMemory) -> str:
    """
    Process a user inquiry through the full RAG pipeline
    
    Pipeline stages:
    1. Retrieval: Fetch relevant logs from database
    2. Evaluation: LLM decides if more data needed
    3. Refinement: Optional second search with refined query
    4. Synthesis: LLM generates technical report
    5. Memory Update: Persist new faults to state
    
    Args:
        user_question: User's inquiry about radar system
        memory_engine: ContinuumMemory instance for state persistence
        
    Returns:
        Technical report or error message
    """
    try:
        print(f"\n[User Inquiry]: {user_question}")

        retrieved_logs = hybrid_retrieve(user_question)
        
        if not retrieved_logs:
            print("[*] Using fallback: returning all logs for analysis")
            # Fallback: get all logs if search fails
            try:
                all_data = table.to_pandas()
                if not all_data.empty:
                    all_results = all_data.head(10).to_dict('records')
                    retrieved_logs = [
                        {
                            "timestamp": r.get("timestamp", ""),
                            "radar_id": r.get("radar_id", ""),
                            "azimuth": r.get("azimuth", 0),
                            "elevation": r.get("elevation", 0),
                            "voltage": r.get("voltage_mv", 0),
                            "error_code": r.get("error_code", ""),
                            "message": r.get("log_message", "")
                        } for r in all_results
                    ]
                else:
                    return "[ERROR] No logs available in database."
            except Exception as e:
                return f"[ERROR] Fallback retrieval failed: {str(e)}"

        # Stage 1: LLM Evaluation - Determine if more data needed
        evaluation_prompt = f"""You are an automated Radar Telemetry Critic. Analyze the user inquiry and the initial retrieved data.

User Query: {user_question}
Initial Results: {json.dumps(retrieved_logs)}

Respond in JSON format:
{{"needs_more_data": true/false, "refined_search_query": ""}}"""

        eval_output = call_local_qwen(evaluation_prompt, json_mode=True)
        
        try:
            eval_json = json.loads(eval_output)
        except json.JSONDecodeError:
            eval_json = {"needs_more_data": False, "refined_search_query": ""}

        # Stage 2: Conditional Refinement
        if eval_json.get("needs_more_data") and eval_json.get("refined_search_query"):
            refined_query = eval_json['refined_search_query']
            print(f"[*] Refined Query: '{refined_query}'")
            secondary_logs = hybrid_retrieve(refined_query, limit=2)
            retrieved_logs.extend([log for log in secondary_logs if log not in retrieved_logs])

        # Stage 3: LLM Synthesis - Generate final report
        final_synthesis_prompt = f"""You are an expert military radar diagnostics model. Synthesize a technical report.

[Memory]: {memory_engine.get_context_string()}
[Logs]: {json.dumps(retrieved_logs, indent=2)}
[Request]: {user_question}

Provide a clear technical summary."""

        response = call_local_qwen(final_synthesis_prompt)
        
        if response.startswith("Error"):
            return f"[ERROR] LLM failed: {response}"

        # Stage 4: Update Memory - Track new faults
        for log in retrieved_logs:
            error_code = log.get("error_code", "")
            radar_id = log.get("radar_id", "")
            if error_code and error_code != "OK" and radar_id:
                memory_engine.update_state(error_code, radar_id)

        return response
        
    except Exception as e:
        return f"[ERROR] process_radar_inquiry failed: {str(e)}"


if __name__ == "__main__":
    print("\n" + "="*70)
    print("🛰️  OFFLINE RADAR TELEMETRY ANALYZER")
    print("Retrieval-Augmented Generation (RAG) Pipeline")
    print("="*70)
    
    # Initialize memory
    radar_memory = ContinuumMemory()
    print("[✔] Continuum Memory initialized")
    
    # Data Input Menu
    data_loaded = False
    while True:
        print("\n" + "-"*70)
        print("📥 DATA INPUT OPTIONS")
        print("-"*70)
        print("1. Interactive Input (type records one by one)")
        print("2. Load from JSON File")
        print("3. View Input Format")
        print("4. Skip Input & Analyze Data")
        print("5. Display Database Stats")
        print("6. Exit Program")
        print("-"*70)
        
        choice = input("Select option (1-6): ").strip()
        
        if choice == "1":
            interactive_input_mode()
            display_database_stats()
            data_loaded = True
        
        elif choice == "2":
            json_file_input_mode()
            display_database_stats()
            data_loaded = True
        
        elif choice == "3":
            print_input_format()
        
        elif choice == "4":
            print("\n[*] Proceeding to data analysis...")
            
            # Check if database has data
            try:
                check_data = table.to_pandas()
                if check_data.empty:
                    print("[⚠️ WARNING] Database is empty!")
                    print("[*] Please load telemetry data first (Option 1 or 2).\n")
                    continue
                data_loaded = True
            except:
                print("[⚠️ WARNING] Could not check database.\n")
                continue
            
            break
        
        elif choice == "5":
            display_database_stats()
        
        elif choice == "6":
            print("\n[✔] Thank you for using Radar Telemetry Analyzer!")
            print("[✔] Exiting program...")
            sys.exit(0)
        
        else:
            print("[ERROR] Invalid option. Please select 1-6.")
    
    # Analysis Mode - MAIN FEATURE (only if data exists)
    if not data_loaded:
        print("[ERROR] No data loaded. Exiting.")
        sys.exit(1)
    
    print("\n" + "="*70)
    print("📊 ANALYSIS MODE")
    print("="*70)
    
    # Generate automatic summary
    print("\n[*] Generating human-readable summary of telemetry data...")
    print("[*] This may take 10-30 seconds...\n")
    summary = generate_human_readable_summary(radar_memory)
    
    print("\n" + "="*70)
    print("📋 TELEMETRY SUMMARY")
    print("="*70)
    print(summary)
    print("="*70)
    
    # Update memory with discovered faults
    try:
        all_data = table.to_pandas()
        if not all_data.empty:
            for error_code in all_data['error_code'].unique():
                if error_code and error_code != "OK":
                    # Get a representative radar_id
                    radar_id = all_data[all_data['error_code'] == error_code]['radar_id'].iloc[0]
                    radar_memory.update_state(error_code, radar_id)
    except:
        pass
    
    # Menu for additional options
    while True:
        print("\n" + "-"*70)
        print("🔧 POST-ANALYSIS OPTIONS")
        print("-"*70)
        print("1. View Error Code Details & Severity")
        print("2. Ask Custom Query (RAG Mode)")
        print("3. Display Database Stats")
        print("4. Generate New Summary")
        print("5. Exit Program")
        print("-"*70)
        
        choice = input("Select option (1-5): ").strip()
        
        if choice == "1":
            display_error_code_details()
        
        elif choice == "2":
            print("\n" + "="*70)
            print("❓ CUSTOM QUERY MODE")
            print("="*70)
            print("Ask questions about your radar telemetry data.")
            print("Type 'back' to return to analysis menu.\n")
            
            while True:
                inquiry = input("\n[Query] Enter your inquiry: ").strip()
                
                if inquiry.lower() == "back":
                    print("\n[*] Returning to analysis menu...")
                    break
                
                if not inquiry:
                    print("[ERROR] Please enter a valid inquiry")
                    continue
                
                print("\n[*] Processing inquiry...")
                print("[*] This may take 10-30 seconds...\n")
                report = process_radar_inquiry(inquiry, radar_memory)
                print("\n" + "="*70)
                print("📋 QUERY RESPONSE")
                print("="*70)
                print(report)
                print("="*70)
        
        elif choice == "3":
            display_database_stats()
        
        elif choice == "4":
            print("\n[*] Generating new summary...")
            print("[*] This may take 10-30 seconds...\n")
            summary = generate_human_readable_summary(radar_memory)
            print("\n" + "="*70)
            print("📋 TELEMETRY SUMMARY (UPDATED)")
            print("="*70)
            print(summary)
            print("="*70)
        
        elif choice == "5":
            print("\n[✔] Thank you for using Radar Telemetry Analyzer!")
            print("[✔] Memory state saved to radar_continuum_memory.json")
            sys.exit(0)
        
        else:
            print("[ERROR] Invalid option. Please select 1-5.")


🛰️  OFFLINE RADAR TELEMETRY ANALYZER
Retrieval-Augmented Generation (RAG) Pipeline
[✔] Continuum Memory initialized

----------------------------------------------------------------------
📥 DATA INPUT OPTIONS
----------------------------------------------------------------------
1. Interactive Input (type records one by one)
2. Load from JSON File
3. View Input Format
4. Skip Input & Analyze Data
5. Display Database Stats
6. Exit Program
----------------------------------------------------------------------


/var/folders/x1/wk2m5xyj59b6qqyrrvjv1bhh0000gn/T/ipykernel_17094/3640129084.py:94: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  if table_name in db.table_names():



📄 JSON FILE INPUT MODE
Expected format: Array of telemetry records
Type 'format' to display expected format.

[✔] 20 records added to database

[✔] Successfully loaded 20 records from /Users/arnav/App_development/LogDataSummarizer/telemetry_test_dataset.json

📊 DATABASE STATISTICS
Total Records: 142
Radar Units: ['', '6', 'AN-FPS-117_A', 'AN-FPS-117_B', 'AN-FPS-117_C']
Error Codes Found: ['6', '', 'OK', 'ERR_403_UNDERVOLT', 'ERR_701_MOTOR_FAULT', 'ERR_501_ALIGN_FAIL', 'ERR_601_THERMAL_HIGH', 'ERR_801_COMMS_LOSS', 'ERR_999_UNKNOWN']
Time Range:  to 6


----------------------------------------------------------------------
📥 DATA INPUT OPTIONS
----------------------------------------------------------------------
1. Interactive Input (type records one by one)
2. Load from JSON File
3. View Input Format
4. Skip Input & Analyze Data
5. Display Database Stats
6. Exit Program
----------------------------------------------------------------------

[*] Proceeding to data analysis...

📊 ANALYS

SystemExit: 0

/Users/arnav/miniconda3/envs/myenv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
#Ma'am's Old RAG Model Code
import re
from datetime import datetime
import ollama


# ---------- READ LOG FILE ----------
def read_logs(file_path):
    with open(file_path, "r") as f:
        return [line.strip() for line in f.readlines()]

# ---------------------------
# Helper Functions
# ---------------------------
def parse_time(t):
    return datetime.strptime(t, "%H:%M:%S")

def time_to_str(t):
    return t.strftime("%H:%M:%S") if t else "N/A"

# ---------------------------
# Input Logs
# ---------------------------
logs = [
"time: 08:30:30 transmitter switched on",
"time: 08:30:38 transmitter put to low power and initialization success",
"time: 08:30:40 Track AR001 put to WTM",
"time: 08:30:45 transmitter freq is 10 code1 20 and code 30",
"time: 08:30:56 transmitter put to high power",
"time: 08:31:20 CTN-AR001, Target id 17,lat 11.18,lon 77.22",
"time: 08:31:25 CTN-AR001, Target id 17,lat 11.20,lon 77.25",
"time: 08:31:50 CTN-AR001, Target id 17,lat 11.24,lon 77.30",
"time: 08:32:05 CTN-AR001, Target id 17,lat 11.28,lon 77.35",
"time: 08:32:20 CTN-AR001, Target id 17,lat 11.30,lon 77.40",
"time: 08:32:30 CTN-AR001, Target id 17,lat 11.32,lon 77.45",
"time: 08:32:37 CTN-AR001, Target id 17,lat 11.34,lon 77.55",
"time: 08:32:39 CTN-AR001, Target id 17,lat 11.38,lon 78.25",
"time: 08:32:40 CTN-AR001, Target id 17,lat 11.42,lon 78.35",
"time: 08:32:48 CTN-AR001, Target id 17,lat 11.46,lon 78.45",
"time: 08:32:50 CTN-AR001, Target id 17,lat 11.50,lon 78.55",
"time: 08:32:58 CTN-AR001, Target id 17,lat 11.53,lon 79.15",
"time: 08:33:20 CTN-AR001, Target id 17,lat 11.55,lon 79.22",
"time: 08:33:50 CTN-AR001, Target id 17,lat 11.57,lon 79.29",
"time: 08:33:59 CTN-AR001, Target id 17,lat 12.20,lon 79.40",
"time: 08:34:30 CTN-AR001, Target id 17,lat 12.25,lon 79.45",
"time: 08:34:39 CTN-AR001, Target id 17,lat 12.28,lon 79.47",
"time: 08:34:40 CTN-AR001, Target id 17,lat 12.30,lon 79.49",
"time: 08:34:50 CTN-AR001, Target id 17,lat 12.36,lon 79.51",
"time: 08:34:56 CTN-AR001, Target id 17,lat 12.40,lon 79.55",
"time: 08:35:25 CTN-AR001, Target id 17,lat 12.45,lon 79.58",
"time: 08:35:30 CTN-AR001, Target id 17,lat 12.50,lon 79.60",
"time: 08:35:35 CTN-AR001, Target id 17,lat 12.55,lon 80.12",
"time: 08:35:40 CTN-AR001, Target id 17,lat 13.10,lon 80.18",
"time: 08:35:45 CTN-AR001, Target id 17,lat 13.20,lon 80.22",
"time: 08:35:50 CTN-AR001, Target id 17,lat 13.30,lon 80.28",
"time: 08:35:54 CTN-AR001, Target id 17,lat 13.40,lon 80.33",
"time: 08:35:55 CTN-AR001, Target id 17,lat 13.50,lon 80.36",
"time: 08:35:59 CTN-AR001, Target id 17,lat 13.58,lon 80.39",
"time: 08:36:30 CTN-AR001, Target id 17,lat 14.33,lon 80.43",
"time: 08:36:39 CTN-AR001, Target id 17,lat 14.39,lon 80.45",
"time: 08:36:40 CTN-AR001, Target id 17,lat 14.40,lon 80.49",
"time: 08:36:45 CTN-AR001, Target id 17,lat 14.43,lon 80.50",
"time: 08:36:49 CTN-AR001, Target id 17,lat 14.50,lon 80.55",
"time: 08:36:50 CTN-AR001, Target id 17,lat 14.57,lon 80.59",
"time: 08:36:58 CTN-AR001, Target id 17,lat 15.22,lon 81.22",
"time: 08:37:10 CTN-AR001, Target id 17,lat 15.30,lon 81.25",
"time: 08:37:20 CTN-AR001, Target id 17,lat 15.33,lon 81.29",
"time: 08:37:30 CTN-AR001, Target id 17,lat 15.37,lon 81.30",
"time: 08:37:55 CTN-AR001, Target id 17,lat 15.39,lon 81.45",
"time: 08:37:59 CTN-AR001, Target id 17,lat 15.44,lon 81.49",
"time: 08:38:10 CTN-AR001, Target id 17,lat 15.49,lon 81.50",
"time: 08:38:30 CTN-AR001, Target id 17,lat 15.50,lon 81.58",
"time: 08:38:35 Track AR002 put to WTM",
"time: 08:38:40 CTN-AR001, Target id 17,lat 15.58,lon 82.35",
"time: 08:38:45 CTN-AR001, Target id 17,lat 16.30,lon 82.39",
"time: 08:38:59 CTN-AR001, Target id 17,lat 16.40,lon 82.45",
"time: 08:39:20 transmitter put to low power",
"time: 08:39:48 transmitter power off"
]


# ---------------------------
# Parsing
# ---------------------------
parsed = []

for line in logs:
    match = re.search(r"time:\s(\d{2}:\d{2}:\d{2})", line)
    if match:
        parsed.append({
            "time": parse_time(match.group(1)),
            "text": line.lower()
        })



# ---------------------------
# Extract Information
# ---------------------------
start_time = None
low_power_times = []
high_power_times = []
init_status = "UNKNOWN"
tracks = set()
frequency = None
targets = {}
shutdown_time = None

for log in parsed:
    t = log["time"]
    text = log["text"]

    if "switched on" in text:
        start_time = t

    if "low power" in text:
        low_power_times.append(time_to_str(t))

    if "high power" in text:
        high_power_times.append(time_to_str(t))

    if "initialization success" in text:
        init_status = "SUCCESS"
    elif "initialization fail" in text:
        init_status = "FAIL"

    if "power off" in text:
        shutdown_time = t

    # Track IDs
    track_match = re.search(r"track (ar\d+)", text)
    if track_match:
        tracks.add(track_match.group(1).upper())

    # Frequency
    freq_match = re.search(r"freq.*code1 (\d+).*code ?(\d+)", text)
    if freq_match:
        frequency = f"code1: {freq_match.group(1)}, code2: {freq_match.group(2)} at {time_to_str(t)}"

    # Target
    target_match = re.search(r"target id (\d+),lat ([\d.]+),lon ([\d.]+)", text)
    if target_match:
        tid = target_match.group(1)

        if tid not in targets:
            targets[tid] = []

        targets[tid].append(
            f"{time_to_str(t)} → ({target_match.group(2)}, {target_match.group(3)})"
        )

# ---------------------------
# Gap Detection (>10 sec)
# ---------------------------
gaps = []

for i in range(1, len(parsed)):
    diff = (parsed[i]["time"] - parsed[i-1]["time"]).seconds
    if diff > 10:
        gaps.append(
            f"{time_to_str(parsed[i-1]['time'])} → {time_to_str(parsed[i]['time'])} ({diff}s)"
        )







# ---------- GAP SENTENCE (SEPARATE GAPS ONLY) ----------
def build_gap_sentence(gaps):
    if not gaps:
        return "No data gaps were observed."

    # ✅ Each gap separate (no merge, no chain)
    gap_list = ", ".join([f"{start}→{end}" for start, end in gaps])

    return f"The system experienced data gaps at {gap_list}."


# ---------- AI PROMPT ----------
def build_prompt(start_time, end_time, gaps):
    

    gaps_text = ", ".join(gaps) if gaps else "No data gaps"
    
    prompt = f"""
Generate a clean line-by-line English summary.

Strict Rules:
- Each point must be exactly one line
- Do not add extra explanation
- Follow this exact order

Format:

Start Time: ...
Power Modes: ...
Initialization Status: ...
Tracks in WTM: ...
Frequency: ...
Target Tracking: ...
Data Gaps: ...
Shutdown Time: ...

Data:
Start Time: {time_to_str(start_time)}
Low Power: {low_power_times}
High Power: {high_power_times}
Initialization: {init_status}
Tracks: {list(tracks)}
Frequency: {frequency}
Targets: {targets}
Gaps: {gaps_text}
Shutdown: {time_to_str(shutdown_time)}
"""

    return prompt


# ---------- AI SUMMARY ----------
def generate_summary(prompt):
    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    return response.get("message", {}).get("content", "").strip()


# ---------- MAIN ----------
if __name__ == "__main__":
    file_path = "log_sheet.txt"

    print("Reading logs...\n")
    lines = read_logs(file_path)

    





    # ---------- AI SUMMARY ----------
    summary = generate_summary(build_prompt(start_time, shutdown_time, gaps))

    # ---------- GAP SENTENCE ----------
    

    # ---------- FINAL OUTPUT ----------
    print("\n------ FINAL SUMMARY ------\n")
    print(summary)
    